**Lab type:** review  
**Course:** NL301 Natural Language Processing with Python  
**Lesson:** 07 — Word and Sentence Embeddings  
**Task:** Review a hybrid retrieval system and answer five questions about embedding behaviour and score comparability.

## Setup

In [ ]:
!pip install sentence-transformers scikit-learn numpy --quiet
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

# Mixed corpus: some short docs, some long docs
short_docs = [
    "Python error",
    "FastAPI install",
    "pandas merge",
]
long_docs = [
    "When training a machine learning model with scikit-learn, always use a Pipeline to avoid data leakage between preprocessing and cross-validation folds.",
    "Sentence transformers encode variable-length text into fixed-size dense vectors using a pretrained transformer with mean pooling over the final hidden states.",
    "FAISS is a library for efficient similarity search and clustering of dense vectors, developed by Meta AI for billion-scale nearest neighbour retrieval.",
]
corpus = short_docs + long_docs
query  = "how to install FastAPI"


## The system to review

In [ ]:
# Hybrid system under review — read before answering the questions below

# TF-IDF scores for short docs
tv = TfidfVectorizer()
tfidf_matrix = tv.fit_transform(short_docs)
tfidf_query  = tv.transform([query])
tfidf_scores = cosine_similarity(tfidf_query, tfidf_matrix)[0]

# Sentence embedding scores for long docs
corpus_embs = model.encode(long_docs, normalize_embeddings=True)
query_emb   = model.encode([query],   normalize_embeddings=True)
emb_scores  = cosine_similarity(query_emb, corpus_embs)[0]

# Combined ranking by raw score — BUG: scores come from different spaces
all_scores = list(zip(short_docs + long_docs,
                      list(tfidf_scores) + list(emb_scores)))
all_scores.sort(key=lambda x: x[1], reverse=True)
print("Combined ranking:")
for doc, score in all_scores:
    print(f"  {score:.4f}  {doc[:70]}")


---
## Review Question 1: Are TF-IDF and embedding scores comparable?

Print the score range for TF-IDF and for sentence embeddings on the same query. Are they on the same scale? What happens when you rank them together?

In [ ]:
print(f"TF-IDF scores:     min={tfidf_scores.min():.4f}  max={tfidf_scores.max():.4f}")
print(f"Embedding scores:  min={emb_scores.min():.4f}   max={emb_scores.max():.4f}")
# Explain: are these comparable? What would a fair combination strategy look like?


---
## Review Question 2: Sentence embeddings and word order

Are `"Python installation error"` and `"error installation Python"` identical under sentence embeddings? They shouldn't be — demonstrate this.

In [ ]:
a = "Python installation error"
b = "error installation Python"
emb_a = model.encode([a], normalize_embeddings=True)
emb_b = model.encode([b], normalize_embeddings=True)
sim = cosine_similarity(emb_a, emb_b)[0][0]
print(f"Cosine similarity: {sim:.4f}")
print("Are they identical embeddings?", np.allclose(emb_a, emb_b))
# If they were averaged word embeddings (GloVe-style), what would the similarity be?


---
## Review Question 3: Out-of-vocabulary handling for 'FastAPI'

If the corpus contains `"FastAPI install"`, how does `all-MiniLM-L6-v2` handle the token `"FastAPI"` vs a GloVe-style fixed vocabulary model? Demonstrate by checking TF-IDF vs embedding similarity for a FastAPI query.

In [ ]:
fastapi_query = "FastAPI web framework"
docs_with_fastapi = ["FastAPI install guide", "flask web application"]

# TF-IDF — what happens if 'FastAPI' is OOV in the fitted vocabulary?
tv2 = TfidfVectorizer()
tv2.fit(["flask web application"])  # fit without FastAPI
q_vec = tv2.transform([fastapi_query])
print("'fastapi' in TF-IDF vocab:", 'fastapi' in tv2.vocabulary_)
print("Non-zero features for query:", q_vec.nnz)

# Sentence embeddings — subword tokenisation handles OOV
emb_docs  = model.encode(docs_with_fastapi, normalize_embeddings=True)
emb_query = model.encode([fastapi_query],   normalize_embeddings=True)
sims = cosine_similarity(emb_query, emb_docs)[0]
for doc, s in zip(docs_with_fastapi, sims):
    print(f"  {s:.4f}  {doc}")


---
## Review Question 4: Model selection for asymmetric retrieval

`all-MiniLM-L6-v2` vs `multi-qa-MiniLM-L6-cos-v1` — what is the difference for asymmetric retrieval where the query is short and documents are long?

In [ ]:
# Load the retrieval-optimised model and compare
try:
    model_qa = SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')
    long_doc = ("Sentence transformers encode variable-length text into fixed-size dense vectors "
                "using a pretrained transformer with mean pooling over the final hidden states.")
    short_query = "how do sentence transformers work"
    emb_gen = model.encode([short_query, long_doc],    normalize_embeddings=True)
    emb_qa  = model_qa.encode([short_query, long_doc], normalize_embeddings=True)
    print(f"all-MiniLM similarity:    {cosine_similarity([emb_gen[0]], [emb_gen[1]])[0][0]:.4f}")
    print(f"multi-qa-MiniLM similarity: {cosine_similarity([emb_qa[0]], [emb_qa[1]])[0][0]:.4f}")
except Exception as e:
    print(f"Model load error: {e}")
# When would you choose one over the other?


---
## Review Question 5: Sentence embeddings on very short documents

For 2–3 word short docs like `"Python error"`, what do you lose by using sentence embeddings vs TF-IDF? When is TF-IDF actually the better choice for short queries?

In [ ]:
short_queries = ["Python error", "pandas merge", "install"]
long_query    = "I am getting a Python AttributeError when I try to merge two pandas dataframes"

for q in short_queries + [long_query]:
    emb = model.encode([q], normalize_embeddings=True)
    sims = cosine_similarity(emb, corpus_embs)[0]
    best = long_docs[np.argmax(sims)]
    print(f"Query: '{q[:50]}'")
    print(f"  Best match: '{best[:70]}'  sim={sims.max():.4f}")
    print()
# Explain: for which query lengths/types does TF-IDF outperform sentence embeddings?
